# 01 - Train the models
Trains (and saves to Drive):
1. **Lesion localizer** (U-Net): finds where the lesion is. Used by the sender (to prioritize blocks) and by the receiver (to find the lesion to classify). A *clean* version and a *robust* version (trained with simulated packet loss).
2. **Lesion classifier** (ResNet-18 on lesion patches, benign vs malignant): `ensemble_size` independent copies of the clean and of the robust variant (the ensemble is used for uncertainty).
3. Baseline results on the test set (no packet loss).

Time on a T4 GPU, a few hundred to ~2,000 images: localizers ~10-30 min each, classifiers ~3-8 min each. Each model is saved as soon as it is trained, so you can stop and resume (cells skip models that already exist).

In [ ]:
# ---- 1. connect Drive, install, load the code ----
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Not running in Colab:', e)
!pip -q install pydicom
import sys, os, numpy as np, pandas as pd, matplotlib.pyplot as plt
CODE_DIR = '/content/drive/MyDrive/mammo_semcom'          # <-- the folder you uploaded (contains src/ and notebooks/)
sys.path.insert(0, CODE_DIR)
from src.config import Cfg

cfg = Cfg(
    cbis_root = '/content/drive/MyDrive/CBIS-DDSM',        # raw data folder: NOT needed after notebook 00 (the cache is used)
    work_dir  = '/content/drive/MyDrive/mammo_semcom_work', # <-- all caches, models, results and figures go here
    max_images = 0,                                         # 0 = everything; try 150 first for a quick dry run
).ensure()
cfg.save()
print(cfg.work_dir)

Mounted at /content/drive
/content/drive/MyDrive/mammo_semcom_work


In [ ]:
from src import data, viz, train as T
from src.pipeline import Receiver, diagnose_all
from src.metrics import *
images, masks, images_df, abn_df = data.load_cache(cfg)
tr_idx = images_df[images_df.split == 'train'].img_idx.values
va_idx = images_df[images_df.split == 'val'].img_idx.values
abn_tr = abn_df[abn_df.split == 'train'].reset_index(drop=True)
abn_va = abn_df[abn_df.split == 'val'].reset_index(drop=True)
abn_te = abn_df[abn_df.split == 'test'].reset_index(drop=True)
te_df = images_df[(images_df.split == 'test') & (~images_df.mixed)].reset_index(drop=True)   # image-level test set
print(len(tr_idx), 'train /', len(va_idx), 'val images |', len(te_df), 'test images | device:', T.DEVICE)

106 train / 22 val images | 22 test images | device: cuda


## 2. Train the lesion localizers

In [ ]:
def have(kind, name): return os.path.exists(cfg.path('models', f'{kind}_{name}.pt'))
for name, robust in [('clean', False), ('robust', True)]:
    if have('localizer', name):
        print('localizer', name, 'already trained - skipping'); continue
    print('=== training localizer:', name)
    T.train_localizer(cfg, images, masks, tr_idx, va_idx, robust=robust, name=name)

=== training localizer: clean
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


  epoch  1/20 | loss 1.6746 | val hit-rate 0.136 | val Dice 0.025
  epoch  2/20 | loss 1.5420 | val hit-rate 0.182 | val Dice 0.042
  epoch  3/20 | loss 1.4842 | val hit-rate 0.182 | val Dice 0.052
  epoch  4/20 | loss 1.4505 | val hit-rate 0.136 | val Dice 0.027
  epoch  5/20 | loss 1.4084 | val hit-rate 0.091 | val Dice 0.030
  epoch  6/20 | loss 1.4154 | val hit-rate 0.409 | val Dice 0.106
  epoch  7/20 | loss 1.3826 | val hit-rate 0.318 | val Dice 0.057
  epoch  8/20 | loss 1.3356 | val hit-rate 0.318 | val Dice 0.029
  epoch  9/20 | loss 1.3273 | val hit-rate 0.273 | val Dice 0.056
  epoch 10/20 | loss 1.3046 | val hit-rate 0.136 | val Dice 0.036
  epoch 11/20 | loss 1.2834 | val hit-rate 0.182 | val Dice 0.053
  epoch 12/20 | loss 1.2142 | val hit-rate 0.227 | val Dice 0.089
  epoch 13/20 | loss 1.1787 | val hit-rate 0.273 | val Dice 0.060
  epoch 14/20 | loss 1.1455 | val hit-rate 0.273 | val Dice 0.057
  epoch 15/20 | loss 1.1232 | val hit-rate 0.318 | val Dice 0.086
  epoch 16

## 3. Train the lesion classifiers (clean and robust ensembles)

In [ ]:
P_va = T.gt_patches(images, abn_va, cfg); y_va = abn_va.label.values
print('train lesions:', len(abn_tr), '| val lesions:', len(abn_va), '| malignant fraction (train):', round(abn_tr.label.mean(), 3))
for variant, robust in [('clean', False), ('robust', True)]:
    for k in range(cfg.ensemble_size):
        name = f'{variant}_{k}'
        if have('classifier', name):
            print('classifier', name, 'already trained - skipping'); continue
        print('=== training classifier:', name)
        m, best = T.train_patch_classifier(cfg, images, abn_tr, P_va, y_va, seed=cfg.seed + k, robust=robust, name=name)
        print('    best validation AUC:', round(best, 3))

train lesions: 115 | val lesions: 33 | malignant fraction (train): 0.391
=== training classifier: clean_0
  epoch  1/15 | train loss 0.9028 | val AUC 0.5055
  epoch  2/15 | train loss 0.8160 | val AUC 0.4890
  epoch  3/15 | train loss 0.7799 | val AUC 0.5055
  epoch  4/15 | train loss 0.7240 | val AUC 0.5110
  epoch  5/15 | train loss 0.7644 | val AUC 0.5000
  epoch  6/15 | train loss 0.7845 | val AUC 0.4670
  epoch  7/15 | train loss 0.7040 | val AUC 0.4835
  epoch  8/15 | train loss 0.6467 | val AUC 0.4451
  epoch  9/15 | train loss 0.6300 | val AUC 0.4835
  epoch 10/15 | train loss 0.5346 | val AUC 0.4945
  epoch 11/15 | train loss 0.5720 | val AUC 0.5385
  epoch 12/15 | train loss 0.5244 | val AUC 0.5220
  epoch 13/15 | train loss 0.5202 | val AUC 0.5330
  epoch 14/15 | train loss 0.5404 | val AUC 0.5165
  epoch 15/15 | train loss 0.5509 | val AUC 0.5055
    best validation AUC: 0.538
=== training classifier: clean_1
  epoch  1/15 | train loss 1.0219 | val AUC 0.5604
  epoch  2/15 

## 4. Baseline on the test set (no packet loss)
* **Classifier only:** patches centred on the true lesion (upper bound for the classifier).
* **End-to-end:** the full receiver pipeline (localize, crop, classify) on the clean mammogram. This is the number every packet-loss result is compared against.

In [ ]:
loc = T.load_localizer(cfg, 'clean'); clfs = [T.load_classifier(cfg, f'clean_{k}') for k in range(cfg.ensemble_size)]
rows = []
# (a) classifier only, GT-centred lesion patches
P_te = T.gt_patches(images, abn_te, cfg); y_te = abn_te.label.values
for name, pr in [('MC-dropout (1 model)', T.mc_predict(clfs[0], P_te, cfg.mc_passes)['p']),
                 (f'ensemble ({len(clfs)} models)', T.ensemble_predict(clfs, P_te)['p'])]:
    a, lo, hi = bootstrap_ci(auc_safe, y_te, pr, abn_te.patient_id.values, n=1000)
    rows.append(dict(stage='classifier only (true lesion centre)', model=name, n=len(y_te), AUC=a, ci_low=lo, ci_high=hi,
                     sens_at_90spec=sens_at_spec(y_te, pr), pAUC_FPR10=pauc(y_te, pr)))
# (b) end-to-end on clean images
X_te = images[te_df.img_idx.values]; M_te = masks[te_df.img_idx.values]
img01 = np.stack([T.to01(a) for a in X_te]); y_img = te_df.label.values
hit = T.hit_rate(T.predict_heatmaps(loc, X_te), M_te)
for name, models_ in [('MC-dropout (1 model)', [clfs[0]]), (f'ensemble ({len(clfs)} models)', clfs)]:
    r = diagnose_all(Receiver(cfg, loc, models_), img01)
    a, lo, hi = bootstrap_ci(auc_safe, y_img, r['p'], te_df.patient_id.values, n=1000)
    rows.append(dict(stage='end-to-end (localize, crop, classify)', model=name, n=len(y_img), AUC=a, ci_low=lo, ci_high=hi,
                     sens_at_90spec=sens_at_spec(y_img, r['p']), pAUC_FPR10=pauc(y_img, r['p']), localization_hit_rate=hit))
base = pd.DataFrame(rows).round(3); base.to_csv(cfg.path('results', 'baseline.csv'), index=False); base

,stage,model,n,AUC,ci_low,ci_high,sens_at_90spec,pAUC_FPR10,localization_hit_rate
0,classifier only (true lesion centre),MC-dropout (1 model),23,0.598,0.357,0.833,0.167,0.482,NaN
1,classifier only (true lesion centre),ensemble (3 models),23,0.606,0.357,0.844,0.167,0.482,NaN
2,"end-to-end (localize, crop, classify)",MC-dropout (1 model),22,0.777,0.554,0.949,0.273,0.617,0.273
3,"end-to-end (localize, crop, classify)",ensemble (3 models),22,0.851,0.653,0.983,0.545,0.587,0.273


## Done
Read the table: the end-to-end AUC is typically lower than the classifier-only AUC because the localizer sometimes misses the lesion (see `localization_hit_rate`). Published AUCs for CBIS-DDSM vary widely with the protocol; report yours with its confidence interval and the exact split.

Next: `02_transmission_experiments.ipynb`.